# Emotion detection

In [ ]:
import os, re
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2S, EfficientNetB2, ResNet50V2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Auto-find dataset
TRAIN_DIR = '/kaggle/input/datasets/ahmedgamall/emotion-detection/Training_data/Training_data'
TEST_DIR  = '/kaggle/input/datasets/ahmedgamall/emotion-detection/test/test'

IMG_SIZE = 96
BATCH    = 32
CLASSES  = sorted(os.listdir(TRAIN_DIR))
print('Classes :', CLASSES)

In [ ]:
# Load images — keep [0, 255], pretrained models handle their own normalization
X, y = [], []
for label, cls in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        X.append(img)
        y.append(label)

X = np.array(X, dtype=np.float32)      # [0, 255]
y = to_categorical(y, len(CLASSES))
print('X:', X.shape, '  y:', y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Augmentation on training data
aug = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

print('Train:', X_train.shape, '  Val:', X_val.shape)

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
    ]


def train_model(model, base, name, unfreeze_layers=40, epochs_phase1=10, epochs_phase2=20):
    """Two-phase training: freeze → fine-tune."""

    # ── Phase 1: frozen base, train head only ─────────────────────────────────
    print(f'\n=== {name} — Phase 1: training head (base frozen) ===')
    base.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(aug.flow(X_train, y_train, batch_size=BATCH),
              validation_data=(X_val, y_val),
              epochs=epochs_phase1, callbacks=get_callbacks())

    # ── Phase 2: unfreeze last N layers, fine-tune at low LR ─────────────────
    print(f'\n=== {name} — Phase 2: fine-tuning last {unfreeze_layers} layers ===')
    base.trainable = True
    for layer in base.layers[:-unfreeze_layers]:
        layer.trainable = False

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),  # 10x lower LR
                  loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(aug.flow(X_train, y_train, batch_size=BATCH),
                        validation_data=(X_val, y_val),
                        epochs=epochs_phase2, callbacks=get_callbacks())

    best = max(history.history['val_accuracy'])
    print(f'\n{name} final val accuracy: {best*100:.2f}%')
    return best


print('Functions ready.')

# Models